# 🌱 Green Code Benchmark — Member Collection Notebook

This notebook is the **only tool the member needs** for model-response collection.

## Input structure

```text
Green-Code-Collection/
├── member_collection.ipynb
└── dataset/
    └── dataset.json
```

The notebook automatically creates and manages:

```text
.collection/
├── state.json
├── logs/
├── raw/
├── code/
└── submissions/
```

It works in both:

- **Google Colab + Google Drive**
- **Local Jupyter / VS Code**

### Member workflow

**Copy prompt → AI web UI → Copy response → Paste → Save & Next**

No API key is required.

> Never modify benchmark prompts or manually edit generated code.


In [1]:

# ============================================================
# 1. ENVIRONMENT + PROJECT LOCATION
# ============================================================

from pathlib import Path
import os, sys, json, re, shutil, zipfile, hashlib, html, traceback
from datetime import datetime, timezone
from IPython.display import display, Markdown, HTML
import ipywidgets as widgets

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()

if IN_COLAB:
    from google.colab import drive
    print("Google Colab detected.")


    drive.mount("/content/drive", force_remount=False)

    # Auto-discover the project folder.
    drive_root = Path("/content/drive/MyDrive")
    candidates = [
        drive_root / "Green-Code-Collection",
        drive_root / "Green Code Collection",
    ]

    PROJECT_ROOT = next((p for p in candidates if (p / "dataset" / "dataset.json").exists()), None)

    if PROJECT_ROOT is None:
        PROJECT_ROOT = drive_root / "Green-Code-Collection"
        print("\nProject folder was not auto-discovered.")
        print("Expected:", PROJECT_ROOT)
        print("If your folder has another name/path, change PROJECT_ROOT below and rerun this cell.")
else:
    print("Local/Jupyter/VS Code environment detected.")

    # Notebook folder.
    try:
        PROJECT_ROOT = Path.cwd()
    except Exception:
        PROJECT_ROOT = Path(".").resolve()

PROJECT_ROOT = Path(PROJECT_ROOT).resolve()

DATASET_FILE = PROJECT_ROOT / "dataset" / "dataset.json"
INTERNAL_ROOT = PROJECT_ROOT / ".collection"

STATE_FILE = INTERNAL_ROOT / "state.json"
LOG_DIR = INTERNAL_ROOT / "logs"
RAW_DIR = INTERNAL_ROOT / "raw"
CODE_DIR = INTERNAL_ROOT / "code"
SUBMISSIONS_DIR = INTERNAL_ROOT / "submissions"

for p in [INTERNAL_ROOT, LOG_DIR, RAW_DIR, CODE_DIR, SUBMISSIONS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("\nEnvironment :", "COLAB" if IN_COLAB else "LOCAL")
print("Project     :", PROJECT_ROOT)
print("Dataset     :", DATASET_FILE)


Google Colab detected.
Mounted at /content/drive

Environment : COLAB
Project     : /content/drive/MyDrive/Green-Code-Collection
Dataset     : /content/drive/MyDrive/Green-Code-Collection/dataset/dataset.json


In [2]:

# ============================================================
# 2. LOAD DATASET
# ============================================================

if not DATASET_FILE.exists():
    raise FileNotFoundError(
        f"\nDataset not found:\n{DATASET_FILE}\n\n"
        "Required structure:\n"
        "Green-Code-Collection/\n"
        "├── member_collection.ipynb\n"
        "└── dataset/\n"
        "    └── dataset.json"
    )

with open(DATASET_FILE, "r", encoding="utf-8") as f:
    DATASET = json.load(f)

# Supports either:
#   [ {...}, {...} ]
# or
#   { "tasks": [ {...}, {...} ] }
if isinstance(DATASET, dict) and "tasks" in DATASET:
    TASKS = DATASET["tasks"]
elif isinstance(DATASET, list):
    TASKS = DATASET
else:
    raise ValueError(
        "dataset.json must be either a list of tasks or an object containing a 'tasks' list."
    )

TASK_MAP = {str(t["task_id"]): t for t in TASKS}

print(f"Loaded {len(TASKS)} tasks.")

if not TASKS:
    raise ValueError("dataset.json contains no tasks.")


Loaded 25 tasks.


## Required dataset format

The notebook expects each task to contain the `interaction_design` used by your benchmark.

For example:

```json
{
  "task_id": "SR-001",
  "title": "Inverted Index Keyword Search",
  "interaction_design": {
    "ONE_SHOT": {
      "prompt": "..."
    },
    "BUG_FIX": {
      "initial_task": "...",
      "bug_report": "..."
    },
    "FEATURE_ADDITION": {
      "initial_task": "...",
      "feature_request": "..."
    },
    "EDGE_CASE": {
      "initial_task": "...",
      "edge_case": "..."
    },
    "FULL_MULTI_TURN": {
      "turn1_initial": "...",
      "turn2_bug": "...",
      "turn3_feature": "...",
      "turn4_edge_case": "..."
    }
  }
}
```

The notebook does **not** hardcode task prompts.


In [3]:

# ============================================================
# 3. COLLECTION CONFIGURATION
# ============================================================

MODELS = [
    ("gpt", "GPT / GPT-OSS"),
    ("claude", "Claude"),
    ("gemini", "Gemini"),
    ("deepseek", "DeepSeek"),
]

MODEL_KEYS = [x[0] for x in MODELS]

INTERACTIONS = [
    "ONE_SHOT",
    "BUG_FIX",
    "FEATURE_ADDITION",
    "EDGE_CASE",
    "FULL_MULTI_TURN",
]

def now():
    return datetime.now(timezone.utc).isoformat()

def log_event(event, **data):
    entry = {
        "timestamp": now(),
        "event": event,
        **data
    }
    log_file = LOG_DIR / "activity.jsonl"
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

def log_error(event, error, **data):
    entry = {
        "timestamp": now(),
        "event": event,
        "error": str(error),
        **data
    }
    with open(LOG_DIR / "errors.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

log_event("ENVIRONMENT_START", environment="colab" if IN_COLAB else "local")


In [4]:
def empty_state():
    return {
        "version": 1,
        "created_at": now(),
        "updated_at": now(),
        "category": None,
        "current": {
            "task_index": 0,
            "task_id": None,
            "model_index": 0,
            "model": None,
            "interaction_index": 0,
            "interaction": None,
            "turn": 0
        },
        "completed": {},
        "history": []
    }

if STATE_FILE.exists():
    try:
        with open(STATE_FILE, "r", encoding="utf-8") as f:
            STATE = json.load(f)
    except Exception as e:
        log_error("STATE_LOAD_ERROR", e)
        STATE = empty_state()
else:
    STATE = empty_state()

def save_state():
    STATE["updated_at"] = now()
    temp = STATE_FILE.with_suffix(".tmp")
    with open(temp, "w", encoding="utf-8") as f:
        json.dump(STATE, f, indent=2, ensure_ascii=False)
    temp.replace(STATE_FILE)

def task_interactions(task):
    # The actual key in dataset.json is 'interactions' (lowercase)
    design = task.get("interactions", {})
    # Convert keys from lowercase to uppercase to match INTERACTIONS list
    return [x for x in INTERACTIONS if x.lower() in design]

def turns_for(interaction):
    if interaction == "ONE_SHOT":
        return 1
    if interaction in ["BUG_FIX", "FEATURE_ADDITION", "EDGE_CASE"]:
        return 2
    if interaction == "FULL_MULTI_TURN":
        return 4
    raise ValueError(interaction)

def initialize_state():
    if STATE.get("category") is None:
        STATE["category"] = DATASET.get("category") if isinstance(DATASET, dict) else None

    # Recover a valid current position.
    # If task_id is not set, or interaction is None (which happened due to previous error), reinitialize interaction.
    if not STATE["current"].get("task_id") or STATE["current"].get("interaction") is None:
        STATE["current"]["task_index"] = 0
        STATE["current"]["task_id"] = str(TASKS[0]["task_id"])
        STATE["current"]["model_index"] = 0
        STATE["current"]["model"] = MODEL_KEYS[0]

        ints = task_interactions(TASKS[0])
        STATE["current"]["interaction_index"] = 0
        STATE["current"]["interaction"] = ints[0] if ints else None
        STATE["current"]["turn"] = 0

initialize_state()
save_state()

print("State loaded.")
print("State file:", STATE_FILE)

State loaded.
State file: /content/drive/MyDrive/Green-Code-Collection/.collection/state.json


In [5]:
def current_task():
    return TASKS[STATE["current"]["task_index"]]

def current_model():
    return STATE["current"]["model"]

def current_interaction():
    return STATE["current"]["interaction"]

def current_turn():
    return STATE["current"]["turn"]

def get_prompt(task, interaction, turn):
    # Access the actual 'interactions' key (lowercase)
    design = task.get("interactions", {})

    # Convert the interaction type to lowercase for dictionary access
    interaction_key_lower = interaction.lower()

    if not design or interaction is None or interaction_key_lower not in design:
        return "Error: Cannot generate prompt. Task design or interaction type is missing/invalid. Please check the `dataset.json` for the 'interactions' key and valid interaction types."

    item = design[interaction_key_lower]

    if interaction == "ONE_SHOT":
        return item.get("prompt", "Error: 'prompt' key missing for ONE_SHOT interaction.")

    # For multi-turn interactions, prompts are in the 'turns' list
    if interaction in ["BUG_FIX", "FEATURE_ADDITION", "EDGE_CASE", "FULL_MULTI_TURN"]:
        turns_list = item.get("turns")
        if not turns_list:
            return f"Error: 'turns' list missing for {interaction} interaction."
        if turn < len(turns_list):
            return turns_list[turn].get("prompt", f"Error: 'prompt' key missing for turn {turn + 1} of {interaction}.")
        else:
            return f"Error: Invalid turn number {turn} for {interaction} interaction. Expected max {len(turns_list)} turns."

    raise ValueError(f"Unsupported interaction type: {interaction}")

prompt = get_prompt(current_task(), current_interaction(), current_turn())

print("Current prompt generated successfully.")

Current prompt generated successfully.


In [6]:

# ============================================================
# 6. FILE NAMING
# ============================================================

def raw_file(task_id, model, interaction, turn):
    return (
        RAW_DIR
        / model
        / str(task_id)
        / interaction
        / f"turn_{turn+1:02d}.txt"
    )

def code_file(task_id, model, interaction, turn):
    base = CODE_DIR / model / str(task_id)

    if interaction == "ONE_SHOT":
        return base / "ONE_SHOT" / "code.py"

    if interaction == "BUG_FIX":
        return base / "BUG_FIX" / ("initial.py" if turn == 0 else "final.py")

    if interaction == "FEATURE_ADDITION":
        return base / "FEATURE_ADDITION" / ("initial.py" if turn == 0 else "final.py")

    if interaction == "EDGE_CASE":
        return base / "EDGE_CASE" / ("initial.py" if turn == 0 else "final.py")

    if interaction == "FULL_MULTI_TURN":
        return base / "FULL_MULTI_TURN" / f"turn_{turn+1:02d}.py"

    raise ValueError(interaction)

def save_text(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


## ⚠️ Response rule

Paste the **complete model response**.

The notebook will:

1. Save the raw response.
2. Look for fenced Python code.
3. Save the code automatically.

If **no Python code block** is found, it will not create a fake answer.

If **multiple Python blocks** are found, it will stop and ask you to review the response instead of guessing.


In [7]:

# ============================================================
# 7. SAFE CODE EXTRACTION
# ============================================================

PY_BLOCK_RE = re.compile(
    r"```(?:python|py)\s*\n?(.*?)```",
    flags=re.IGNORECASE | re.DOTALL
)

def extract_python_blocks(response):
    return [x.strip() for x in PY_BLOCK_RE.findall(response) if x.strip()]

def save_model_response(response):
    task = current_task()
    tid = str(task["task_id"])
    model = current_model()
    interaction = current_interaction()
    turn = current_turn()

    blocks = extract_python_blocks(response)

    if len(blocks) == 0:
        log_event(
            "NO_CODE_DETECTED",
            task_id=tid,
            model=model,
            interaction=interaction,
            turn=turn + 1
        )
        return False, "No fenced Python code block detected."

    if len(blocks) > 1:
        log_event(
            "MULTIPLE_CODE_BLOCKS",
            task_id=tid,
            model=model,
            interaction=interaction,
            turn=turn + 1,
            count=len(blocks)
        )
        return False, f"{len(blocks)} Python code blocks detected. Review the response."

    rp = raw_file(tid, model, interaction, turn)
    cp = code_file(tid, model, interaction, turn)

    save_text(rp, response)
    save_text(cp, blocks[0])

    # FULL_MULTI_TURN final.py = final response after turn 4.
    if interaction == "FULL_MULTI_TURN" and turn == 3:
        final_path = CODE_DIR / model / tid / "FULL_MULTI_TURN" / "final.py"
        save_text(final_path, blocks[0])

    # Save a small response manifest beside the raw response.
    manifest = {
        "task_id": tid,
        "model": model,
        "interaction": interaction,
        "turn": turn + 1,
        "saved_at": now(),
        "response_sha256": hashlib.sha256(response.encode("utf-8")).hexdigest(),
        "code_sha256": hashlib.sha256(blocks[0].encode("utf-8")).hexdigest(),
        "raw_file": str(rp.relative_to(PROJECT_ROOT)),
        "code_file": str(cp.relative_to(PROJECT_ROOT))
    }

    save_text(
        rp.with_suffix(".json"),
        json.dumps(manifest, indent=2, ensure_ascii=False)
    )

    log_event(
        "RESPONSE_SAVED",
        task_id=tid,
        model=model,
        interaction=interaction,
        turn=turn + 1,
        raw_file=str(rp.relative_to(PROJECT_ROOT)),
        code_file=str(cp.relative_to(PROJECT_ROOT))
    )

    return True, str(cp.relative_to(PROJECT_ROOT))


In [8]:

# ============================================================
# 8. ADVANCE STATE — CRASH SAFE
# ============================================================

def unit_key(task_id, model, interaction):
    return f"{task_id}|{model}|{interaction}"

def mark_turn_complete():
    tid = str(current_task()["task_id"])
    model = current_model()
    interaction = current_interaction()
    turn = current_turn()

    key = unit_key(tid, model, interaction)

    if key not in STATE["completed"]:
        STATE["completed"][key] = {
            "task_id": tid,
            "model": model,
            "interaction": interaction,
            "turns": []
        }

    if turn + 1 not in STATE["completed"][key]["turns"]:
        STATE["completed"][key]["turns"].append(turn + 1)

def is_interaction_complete(task, model, interaction):
    key = unit_key(str(task["task_id"]), model, interaction)
    expected = list(range(1, turns_for(interaction) + 1))
    actual = sorted(STATE["completed"].get(key, {}).get("turns", []))
    return actual == expected

def advance_position():
    mark_turn_complete()

    task = current_task()
    interactions = task_interactions(task)
    interaction = current_interaction()

    # Next turn
    if current_turn() + 1 < turns_for(interaction):
        STATE["current"]["turn"] += 1
        return

    # Next interaction
    if STATE["current"]["interaction_index"] + 1 < len(interactions):
        STATE["current"]["interaction_index"] += 1
        STATE["current"]["interaction"] = interactions[
            STATE["current"]["interaction_index"]
        ]
        STATE["current"]["turn"] = 0
        return

    # Next model
    if STATE["current"]["model_index"] + 1 < len(MODEL_KEYS):
        STATE["current"]["model_index"] += 1
        STATE["current"]["model"] = MODEL_KEYS[
            STATE["current"]["model_index"]
        ]
        STATE["current"]["interaction_index"] = 0
        STATE["current"]["interaction"] = interactions[0]
        STATE["current"]["turn"] = 0
        return

    # Current task complete -> next task
    tid = str(task["task_id"])
    STATE["history"].append({
        "event": "TASK_COMPLETED",
        "task_id": tid,
        "completed_at": now()
    })

    if STATE["current"]["task_index"] + 1 < len(TASKS):
        STATE["current"]["task_index"] += 1
        next_task = current_task()
        next_interactions = task_interactions(next_task)

        STATE["current"]["task_id"] = str(next_task["task_id"])
        STATE["current"]["model_index"] = 0
        STATE["current"]["model"] = MODEL_KEYS[0]
        STATE["current"]["interaction_index"] = 0
        STATE["current"]["interaction"] = next_interactions[0]
        STATE["current"]["turn"] = 0

        log_event("TASK_COMPLETED", task_id=tid)
        return

    # Everything complete
    STATE["current"]["task_id"] = None
    STATE["current"]["interaction"] = None
    STATE["current"]["turn"] = 0
    log_event("ALL_TASKS_COMPLETED", category=STATE.get("category"))

save_state()


In [9]:

# ============================================================
# 9. DASHBOARD + INTERACTION UI
# ============================================================

def completed_task_count():
    count = 0
    for task in TASKS:
        tid = str(task["task_id"])
        all_done = True
        for model in MODEL_KEYS:
            for interaction in task_interactions(task):
                if not is_interaction_complete(task, model, interaction):
                    all_done = False
                    break
            if not all_done:
                break
        if all_done:
            count += 1
    return count

def dashboard_html():
    if STATE["current"]["task_id"] is None:
        current_text = "ALL TASKS COMPLETED"
        model_text = "-"
        interaction_text = "-"
        turn_text = "-"
    else:
        current_text = str(STATE["current"]["task_id"])
        model_text = str(STATE["current"]["model"])
        interaction_text = str(STATE["current"]["interaction"])
        turn_text = f'{STATE["current"]["turn"] + 1}/{turns_for(STATE["current"]["interaction"])}'

    done = completed_task_count()
    total = len(TASKS)
    pct = (done / total * 100) if total else 0

    return f"""
    <div style="
        border:1px solid #d9d9d9;
        border-radius:14px;
        padding:18px;
        margin:10px 0;
        font-family:Arial,sans-serif;">
        <h2 style="margin-top:0">🌱 Green Code Collection</h2>
        <b>Environment:</b> {"Google Colab" if IN_COLAB else "Local / VS Code"}<br>
        <b>Project:</b> {html.escape(str(PROJECT_ROOT))}<br>
        <b>Tasks:</b> {done} / {total}
        <div style="background:#eee;border-radius:8px;height:12px;margin:8px 0 14px;">
            <div style="background:#4caf50;width:{pct:.1f}%;height:12px;border-radius:8px;"></div>
        </div>
        <b>Current Task:</b> {html.escape(current_text)}<br>
        <b>Model:</b> {html.escape(model_text)}<br>
        <b>Interaction:</b> {html.escape(interaction_text)}<br>
        <b>Turn:</b> {html.escape(turn_text)}
    </div>
    """

dashboard = widgets.HTML()
prompt_area = widgets.Textarea(
    layout=widgets.Layout(width="100%", height="240px"),
    disabled=True
)
response_area = widgets.Textarea(
    placeholder="Paste the COMPLETE model response here...",
    layout=widgets.Layout(width="100%", height="320px")
)

copy_btn = widgets.Button(description="📋 COPY PROMPT", button_style="info")
save_btn = widgets.Button(description="💾 SAVE & NEXT", button_style="success")
refresh_btn = widgets.Button(description="🔄 REFRESH", button_style="")
message = widgets.HTML()

def show_message(text, color="#333"):
    message.value = (
        f'<div style="padding:10px;font-weight:600;color:{color}">'
        f'{html.escape(text)}</div>'
    )

def refresh_ui():
    dashboard.value = dashboard_html()

    if STATE["current"]["task_id"] is None:
        prompt_area.value = "All tasks completed."
        response_area.value = ""
        return

    prompt_area.value = get_prompt(
        current_task(),
        current_interaction(),
        current_turn()
    )
    response_area.value = ""

def copy_prompt(_):
    # Browser clipboard permissions vary, so we select the prompt.
    display(HTML("""
    <script>
    const areas = document.querySelectorAll('textarea');
    if (areas.length > 0) {
        const el = areas[areas.length - 2];
        el.focus();
        el.select();
    }
    </script>
    """))
    show_message(
        "Prompt selected. Press Ctrl+C / Cmd+C, then paste it into the AI web UI.",
        "#1565c0"
    )

def save_next(_):
    if STATE["current"]["task_id"] is None:
        show_message("All tasks are already complete.", "#2e7d32")
        return

    response = response_area.value.strip()

    if not response:
        show_message("Paste the complete model response first.", "#b71c1c")
        return

    try:
        ok, result = save_model_response(response)

        if not ok:
            show_message(result, "#b71c1c")
            return

        # Important: save output BEFORE moving state.
        advance_position()
        save_state()

        refresh_ui()
        show_message(
            f"Saved successfully: {result}",
            "#2e7d32"
        )

    except Exception as e:
        log_error(
            "SAVE_NEXT_ERROR",
            e,
            task_id=STATE["current"].get("task_id"),
            model=STATE["current"].get("model"),
            interaction=STATE["current"].get("interaction"),
            turn=STATE["current"].get("turn")
        )
        show_message(
            f"Error: {e}. Progress was not advanced.",
            "#b71c1c"
        )
        traceback.print_exc()

def refresh_click(_):
    refresh_ui()
    show_message("State reloaded from Drive/local storage.", "#1565c0")

copy_btn.on_click(copy_prompt)
save_btn.on_click(save_next)
refresh_btn.on_click(refresh_click)

display(dashboard)
display(Markdown("### 1️⃣ Exact benchmark prompt"))
display(prompt_area)
display(copy_btn)

display(Markdown("### 2️⃣ Paste the complete model response"))
display(response_area)
display(save_btn, refresh_btn)
display(message)

refresh_ui()


HTML(value='')

### 1️⃣ Exact benchmark prompt

Textarea(value='', disabled=True, layout=Layout(height='240px', width='100%'))

Button(button_style='info', description='📋 COPY PROMPT', style=ButtonStyle())

### 2️⃣ Paste the complete model response

Textarea(value='', layout=Layout(height='320px', width='100%'), placeholder='Paste the COMPLETE model response…

Button(button_style='success', description='💾 SAVE & NEXT', style=ButtonStyle())

Button(description='🔄 REFRESH', style=ButtonStyle())

HTML(value='')

## 🔐 Conversation rules

The notebook tracks turns, but **the model's web conversation context must be handled correctly**.

| Interaction | Conversation |
|---|---|
| ONE_SHOT | New conversation |
| BUG_FIX | Same conversation for both turns |
| FEATURE_ADDITION | Same conversation for both turns |
| EDGE_CASE | Same conversation for both turns |
| FULL_MULTI_TURN | Same conversation for all 4 turns |

Do not mix one interaction's conversation with another.


In [10]:

# ============================================================
# 10. PROGRESS VIEW
# ============================================================

def progress_report():
    print("=" * 60)
    print("GREEN CODE COLLECTION PROGRESS")
    print("=" * 60)
    print("Category :", STATE.get("category"))
    print("Completed:", completed_task_count(), "/", len(TASKS))

    if STATE["current"]["task_id"] is None:
        print("STATUS   : ALL TASKS COMPLETED")
    else:
        print("Task     :", STATE["current"]["task_id"])
        print("Model    :", STATE["current"]["model"])
        print("Interact.:", STATE["current"]["interaction"])
        print("Turn     :", STATE["current"]["turn"] + 1,
              "/", turns_for(STATE["current"]["interaction"]))

progress_report()


GREEN CODE COLLECTION PROGRESS
Category : None
Completed: 8 / 25
Task     : FD-009
Model    : claude
Interact.: BUG_FIX
Turn     : 1 / 2


In [11]:

# ============================================================
# 11. FINAL VALIDATION
# ============================================================

def expected_files_for_task(task):
    expected = []
    tid = str(task["task_id"])

    for model in MODEL_KEYS:
        for interaction in task_interactions(task):
            if interaction == "ONE_SHOT":
                expected.append(CODE_DIR/model/tid/"ONE_SHOT"/"code.py")

            elif interaction == "BUG_FIX":
                expected += [
                    CODE_DIR/model/tid/"BUG_FIX"/"initial.py",
                    CODE_DIR/model/tid/"BUG_FIX"/"final.py"
                ]

            elif interaction == "FEATURE_ADDITION":
                expected += [
                    CODE_DIR/model/tid/"FEATURE_ADDITION"/"initial.py",
                    CODE_DIR/model/tid/"FEATURE_ADDITION"/"final.py"
                ]

            elif interaction == "EDGE_CASE":
                expected += [
                    CODE_DIR/model/tid/"EDGE_CASE"/"initial.py",
                    CODE_DIR/model/tid/"EDGE_CASE"/"final.py"
                ]

            elif interaction == "FULL_MULTI_TURN":
                expected += [
                    CODE_DIR/model/tid/"FULL_MULTI_TURN"/"turn_01.py",
                    CODE_DIR/model/tid/"FULL_MULTI_TURN"/"turn_02.py",
                    CODE_DIR/model/tid/"FULL_MULTI_TURN"/"turn_03.py",
                    CODE_DIR/model/tid/"FULL_MULTI_TURN"/"turn_04.py",
                    CODE_DIR/model/tid/"FULL_MULTI_TURN"/"final.py"
                ]
    return expected

def validate_all():
    missing = []
    for task in TASKS:
        for p in expected_files_for_task(task):
            if not p.exists() or not p.read_text(encoding="utf-8").strip():
                missing.append(str(p.relative_to(PROJECT_ROOT)))

    print("Validation")
    print("=" * 60)

    if missing:
        print(f"❌ Missing/empty files: {len(missing)}")
        for p in missing[:100]:
            print(" -", p)
        if len(missing) > 100:
            print(f"... and {len(missing)-100} more")
        return False

    print("✅ All expected generated-code files are present.")
    return True


In [12]:

# ============================================================
# 12. FINAL MANIFEST + ZIP
# ============================================================

def create_manifest():
    files = []
    for p in CODE_DIR.rglob("*"):
        if p.is_file():
            data = p.read_bytes()
            files.append({
                "path": str(p.relative_to(PROJECT_ROOT)),
                "sha256": hashlib.sha256(data).hexdigest(),
                "size_bytes": len(data)
            })

    manifest = {
        "created_at": now(),
        "category": STATE.get("category"),
        "task_count": len(TASKS),
        "models": MODEL_KEYS,
        "interactions": INTERACTIONS,
        "files": files
    }

    manifest_path = SUBMISSIONS_DIR / "manifest.json"
    manifest_path.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    return manifest_path

def create_submission_zip():
    if completed_task_count() != len(TASKS):
        print("❌ Not all tasks are complete.")
        print(f"Completed: {completed_task_count()} / {len(TASKS)}")
        return None

    if not validate_all():
        return None

    manifest_path = create_manifest()

    zip_path = SUBMISSIONS_DIR / "collection_submission.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        # Generated code
        for p in CODE_DIR.rglob("*"):
            if p.is_file():
                z.write(p, p.relative_to(PROJECT_ROOT))

        # Raw responses are included for provenance.
        for p in RAW_DIR.rglob("*"):
            if p.is_file():
                z.write(p, p.relative_to(PROJECT_ROOT))

        # Manifest
        z.write(manifest_path, manifest_path.relative_to(PROJECT_ROOT))

    log_event("SUBMISSION_ZIP_CREATED", zip=str(zip_path.relative_to(PROJECT_ROOT)))

    print("\n✅ Submission created:")
    print(zip_path)

    return zip_path

zip_btn = widgets.Button(
    description="📦 CREATE FINAL SUBMISSION ZIP",
    button_style="success"
)

def zip_click(_):
    try:
        create_submission_zip()
    except Exception as e:
        log_error("ZIP_ERROR", e)
        print("❌ ZIP creation failed:", e)

zip_btn.on_click(zip_click)

display(Markdown("### Final export"))
display(zip_btn)


### Final export

Button(button_style='success', description='📦 CREATE FINAL SUBMISSION ZIP', style=ButtonStyle())

## Final Drive/local output

After collection, the notebook manages everything under:

```text
.collection/
├── state.json
├── logs/
│   ├── activity.jsonl
│   └── errors.jsonl
├── raw/
│   └── <model>/<task>/<interaction>/turn_XX.txt
├── code/
│   └── <model>/<task>/<interaction>/*.py
└── submissions/
    ├── manifest.json
    └── collection_submission.zip
```

You can close Colab, come back days later, reconnect Drive, run the notebook again, and it will resume from the persistent state.

The original `dataset/dataset.json` is never modified by the collection notebook.
